# Notebook 05 — Phase 5: ChemProp D-MPNN + Optuna (Kaggle edition)

**Goal.** Tune a D-MPNN on ESOL using 30 Optuna trials with 5-fold scaffold-disjoint CV on the seed=42 training set. Persist `best_params` for Sera 4 (5-seed outer evaluation).

**Why Kaggle, not Colab.** Colab Free GPU quota ran out mid-Phase 5. Kaggle provides 30 GPU-hours/week + background execution + email notification on completion, which fits a 7–12 hour Optuna study perfectly: launch, close browser, wait for email.

**Protocol** (consistent with Phases 3 & 4):
- **Inner**: 5-fold scaffold-disjoint CV on seed=42 training set → optimize mean valid RMSE across folds.
- **Trial budget**: 30 (smaller than Phase 4's 100, because ChemProp's search space is tighter — Yang et al. 2019 explored 7 hyperparameters; we cover 6 of them).
- **Sampler**: TPE (Optuna default Bayesian).
- **Pruner**: MedianPruner (cuts trials below median early; ~30–40% compute savings).
- **Storage**: SQLite at `/kaggle/working/optuna_chemprop.db` — survives intra-session crashes; cross-session resume via Kaggle Save Version.

**Pre-registered target** (from Sera 1 sanity check + literature):
- **Strong**: mean CV RMSE ≤ 0.70 → ChemProp graph representation is competitive with hand-engineered tabular features (P3 = 0.862, P4 = 1.436).
- **Acceptable**: mean CV RMSE ≤ 0.80 → improvement over the no-tuning baseline (0.99, Sera 1).
- **Concerning**: mean CV RMSE > 0.80 → revisit search space.

Total compute estimate: 30 trials × 5 folds × ~3 min/fit ≈ 7.5 hours on T4.

**Setup checklist**: Kaggle notebook with GPU T4 enabled; `qsar-esol-data` private dataset attached at `/kaggle/input/qsar-esol-data/`.

## 1. Install ChemProp + verify environment

In [ ]:
# Kaggle base image already has PyTorch + Lightning + RDKit.
# Only chemprop needs to be installed.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'chemprop>=2.0,<3.0', 'optuna>=3.5'])

import chemprop
import torch
import lightning
import optuna

print(f'ChemProp  : {chemprop.__version__}')
print(f'PyTorch   : {torch.__version__}')
print(f'Lightning : {lightning.__version__}')
print(f'Optuna    : {optuna.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')

## 2. Setup paths + import `src/` modules

On Kaggle, the project files live read-only at `/kaggle/input/qsar-esol-data/`. Outputs go to `/kaggle/working/` (persisted across the running session, downloadable via Save Version).

In [ ]:
import sys
from pathlib import Path

# Input (read-only) — adjust slug if you used a different dataset name
INPUT_ROOT = Path('/kaggle/input/qsar-esol-data')
OUTPUT_DIR = Path('/kaggle/working')

assert INPUT_ROOT.exists(), f'Input dataset not attached at {INPUT_ROOT}'
assert (INPUT_ROOT / 'src' / '__init__.py').exists(), 'src/ folder missing in dataset'
assert (INPUT_ROOT / 'data' / 'processed' / 'esol_dedup.csv').exists(), 'dataset CSV missing'

# Inject src/ into PYTHONPATH
if str(INPUT_ROOT) not in sys.path:
    sys.path.insert(0, str(INPUT_ROOT))

# Drop any partially-imported 'src' from sys.modules (safety)
for mod_name in list(sys.modules.keys()):
    if mod_name == 'src' or mod_name.startswith('src.'):
        del sys.modules[mod_name]

from src.splits import scaffold_split_balanced, scaffold_kfold
from src.training import seed_everything
from src.chemprop_training import train_chemprop

REPORTS_DIR = OUTPUT_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input root  : {INPUT_ROOT}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Reports dir : {REPORTS_DIR}')

## 3. Load dataset, build scaffold split (seed=42), inner CV folds

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

TUNING_SEED = 42  # the tuning seed — same as Phases 3 and 4
seed_everything(TUNING_SEED)

df = pd.read_csv(INPUT_ROOT / 'data' / 'processed' / 'esol_dedup.csv')
df['mol'] = df['smiles'].apply(Chem.MolFromSmiles)
assert df['mol'].isna().sum() == 0
print(f'Loaded {len(df)} molecules')

# Outer split — train+valid used for tuning; test set untouched (Sera 4 outer eval)
tr_idx, va_idx, te_idx = scaffold_split_balanced(df, seed=TUNING_SEED)
df_tr_outer = df.iloc[tr_idx].reset_index(drop=True)
df_va_outer = df.iloc[va_idx].reset_index(drop=True)
df_te_outer = df.iloc[te_idx].reset_index(drop=True)
print(f'Outer split: train={len(df_tr_outer)}, valid={len(df_va_outer)}, test={len(df_te_outer)}')

# Inner CV: 5-fold scaffold-disjoint on the OUTER TRAINING SET only
# (NOT on train+valid combined — keeps valid set untouched for sanity reference)
cv_folds = scaffold_kfold(df_tr_outer, n_folds=5, seed=TUNING_SEED)
for k, (cv_tr, cv_va) in enumerate(cv_folds):
    print(f'  Fold {k}: cv_train={len(cv_tr)}, cv_valid={len(cv_va)}')

## 4. Optuna search space + objective

**Search space (6 hyperparameters)** — informed by Yang et al. 2019 ESOL exploration and ChemProp v2 defaults:

| Hyperparameter        | Range / values             | Why                                                     |
|-----------------------|---------------------------|---------------------------------------------------------|
| `depth`               | {2, 3, 4, 5}              | Yang 2019 sweet spot 3-5; default 3                     |
| `mp_hidden_dim`       | {200, 300, 400, 500}      | Default 300; small dataset, no need to go higher        |
| `mp_dropout`          | [0.0, 0.4] uniform        | Default 0.0; some dropout helps generalization here     |
| `ffn_hidden_dim`      | {200, 300, 500, 700}      | Default 300; readout MLP capacity                       |
| `ffn_num_layers`      | {1, 2, 3}                 | Default 1 (linear head); deeper FFN sometimes helps     |
| `max_lr`              | [3e-4, 3e-3] log-uniform  | Default 1e-3; Noam scheduler is sensitive to peak LR    |

**Fixed** (not tuned):
- `batch_size` = 50 (default; ESOL is small)
- `init_lr`, `final_lr` = 1e-4 (default warmup/cooldown LR)
- `warmup_epochs` = 2 (default)
- `max_epochs` = 150, `patience` = 30 (from Sera 1 v2 verified setup)

**Pruning**: each fold reports an intermediate value at training end; MedianPruner kills trials below the median of completed trials at the same step.

In [ ]:
import time

def objective(trial: optuna.Trial) -> float:
    """5-fold scaffold-disjoint CV mean valid RMSE for ChemProp."""
    # --- Search space ---
    depth          = trial.suggest_int('depth', 2, 5)
    mp_hidden_dim  = trial.suggest_categorical('mp_hidden_dim', [200, 300, 400, 500])
    mp_dropout     = trial.suggest_float('mp_dropout', 0.0, 0.4)
    ffn_hidden_dim = trial.suggest_categorical('ffn_hidden_dim', [200, 300, 500, 700])
    ffn_num_layers = trial.suggest_int('ffn_num_layers', 1, 3)
    max_lr         = trial.suggest_float('max_lr', 3e-4, 3e-3, log=True)

    fold_rmses = []
    t0 = time.time()
    for k, (cv_tr, cv_va) in enumerate(cv_folds):
        smiles_tr = df_tr_outer.iloc[cv_tr]['smiles'].tolist()
        smiles_va = df_tr_outer.iloc[cv_va]['smiles'].tolist()
        y_tr      = df_tr_outer.iloc[cv_tr]['logS'].values
        y_va      = df_tr_outer.iloc[cv_va]['logS'].values

        try:
            result = train_chemprop(
                smiles_train=smiles_tr, y_train=y_tr,
                smiles_valid=smiles_va, y_valid=y_va,
                depth=depth,
                mp_hidden_dim=mp_hidden_dim,
                mp_dropout=mp_dropout,
                ffn_hidden_dim=ffn_hidden_dim,
                ffn_num_layers=ffn_num_layers,
                max_lr=max_lr,
                batch_size=50,
                max_epochs=150, patience=30,
                seed=TUNING_SEED,
                verbose=False,
            )
            fold_rmse = result['best_valid_rmse']
        except Exception as e:
            print(f'  [trial {trial.number}, fold {k}] failed: {e}')
            raise optuna.TrialPruned()

        fold_rmses.append(fold_rmse)

        # Report intermediate value and check pruning after each fold
        running_mean = float(np.mean(fold_rmses))
        trial.report(running_mean, step=k)
        if trial.should_prune():
            elapsed = time.time() - t0
            print(f'  [trial {trial.number}] PRUNED after fold {k}/{5}  '
                  f'(running mean RMSE = {running_mean:.4f}, {elapsed/60:.1f} min)')
            raise optuna.TrialPruned()

    mean_rmse = float(np.mean(fold_rmses))
    elapsed = time.time() - t0
    print(f'  [trial {trial.number}] CV RMSE = {mean_rmse:.4f} '
          f'(folds: {[round(r,3) for r in fold_rmses]})  {elapsed/60:.1f} min')
    return mean_rmse

print('Objective function ready.')

## 5. Create / resume the Optuna study

`storage='sqlite:///...'` makes the study resumable: if Kaggle session times out (9h cap) or you click Stop, you can re-run this cell and Optuna picks up from where it left off — only the trials that completed are kept, in-progress ones restart.

If the DB file already exists from a previous session, it loads the existing study and continues. If not, a fresh study is created.

In [ ]:
STUDY_NAME = 'chemprop_esol_phase5'
DB_PATH    = OUTPUT_DIR / 'optuna_chemprop.db'
STORAGE    = f'sqlite:///{DB_PATH}'

# Configure logging — make sure Optuna's chatter ends up in the cell output
optuna.logging.set_verbosity(optuna.logging.INFO)

sampler = optuna.samplers.TPESampler(seed=TUNING_SEED)
pruner  = optuna.pruners.MedianPruner(
    n_startup_trials=5,    # don't prune the first 5 trials (let TPE warm up)
    n_warmup_steps=2,      # don't prune before 2 folds are done
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    direction='minimize',
    sampler=sampler,
    pruner=pruner,
    load_if_exists=True,   # resume if DB has the same study_name
)

n_already_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
n_pruned       = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
print(f'Study     : {STUDY_NAME}')
print(f'Storage   : {STORAGE}')
print(f'Trials    : {len(study.trials)} total ({n_already_done} complete, {n_pruned} pruned)')
if n_already_done > 0:
    print(f'Best so far: {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')

## 6. Run trials

Targets `n_target_trials=30` total. If the study already has some completed trials (from a previous session), only the remaining are run.

**To run in background**: click the "Save Version" button (top right) and choose "Save & Run All (Commit)". The notebook executes server-side, you can close the browser, and you'll receive an email at completion. This is the recommended path for the 7–12h study.

In [ ]:
N_TARGET_TRIALS = 30
n_existing = len([t for t in study.trials
                  if t.state in (optuna.trial.TrialState.COMPLETE,
                                 optuna.trial.TrialState.PRUNED)])
n_remaining = max(0, N_TARGET_TRIALS - n_existing)

print(f'Target trials: {N_TARGET_TRIALS}')
print(f'Already done : {n_existing}')
print(f'Remaining    : {n_remaining}')

if n_remaining > 0:
    t_start = time.time()
    study.optimize(
        objective,
        n_trials=n_remaining,
        gc_after_trial=True,  # free GPU memory between trials
        show_progress_bar=False,
    )
    elapsed = time.time() - t_start
    print(f'\nStudy complete. Wall time this session: {elapsed/3600:.2f} h')
else:
    print('All target trials done.')

## 7. Study results — best params + FANOVA importance

FANOVA decomposes the variance of the objective into contributions of each hyperparameter (same diagnostic used in Phases 3 and 4). If one hyperparameter dominates, the rest may be set to defaults in future runs.

In [ ]:
print('=== Study summary ===')
print(f'Total trials   : {len(study.trials)}')
print(f'  - complete   : {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}')
print(f'  - pruned     : {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}')
print(f'  - failed     : {len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])}')
print(f'Best CV RMSE   : {study.best_value:.4f}')
print('\nBest params:')
for k, v in study.best_params.items():
    print(f'  {k:<20} = {v}')

# Pre-registered target check
TARGET_STRONG     = 0.70
TARGET_ACCEPTABLE = 0.80
verdict = (
    'STRONG'     if study.best_value <= TARGET_STRONG
    else 'ACCEPTABLE' if study.best_value <= TARGET_ACCEPTABLE
    else 'CONCERNING'
)
print(f'\nPre-registered verdict: {verdict}')
print(f'  STRONG     ≤ {TARGET_STRONG} (graph beats tabular)')
print(f'  ACCEPTABLE ≤ {TARGET_ACCEPTABLE} (graph beats no-tune baseline 0.99)')
print(f'  Got: {study.best_value:.4f}')

In [ ]:
# FANOVA hyperparameter importance
try:
    importance = optuna.importance.get_param_importances(study)
    print('=== FANOVA importance ===')
    for k, v in sorted(importance.items(), key=lambda x: -x[1]):
        bar = '█' * int(v * 50)
        print(f'  {k:<20} {v*100:5.1f}%  {bar}')
except Exception as e:
    print(f'FANOVA computation failed: {e}')
    print('(Needs ≥2 completed trials with different values for each param.)')

## 8. Persist best params + study metadata

Writes `chemprop_esol_phase5_best_params.json` to `/kaggle/working/`. After Save Version, this file is available in the notebook's output (right sidebar) and can be added as a Kaggle dataset for Sera 4 (5-seed outer evaluation).

In [ ]:
import json

best_params_record = {
    'phase': 'phase5_chemprop_dmpnn',
    'best_params': dict(study.best_params),
    'best_value_cv_rmse': float(study.best_value),
    'tuning_protocol': 'scaffold_kfold_5_seed42',
    'tuning_seed': TUNING_SEED,
    'n_trials_target': N_TARGET_TRIALS,
    'n_complete': len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
    'n_pruned':   len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]),
    'n_failed':   len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]),
    'sampler': 'TPE',
    'pruner':  'MedianPruner(n_startup=5, n_warmup_steps=2)',
    'fixed_params': {
        'batch_size': 50,
        'init_lr': 1e-4,
        'final_lr': 1e-4,
        'warmup_epochs': 2,
        'max_epochs': 150,
        'patience': 30,
    },
    'environment': {
        'chemprop': chemprop.__version__,
        'pytorch': torch.__version__,
        'lightning': lightning.__version__,
        'optuna': optuna.__version__,
        'platform': 'kaggle',
    },
}

best_params_path = OUTPUT_DIR / 'chemprop_esol_phase5_best_params.json'
best_params_path.write_text(json.dumps(best_params_record, indent=2))
print(f'Saved: {best_params_path}')
print(json.dumps(best_params_record, indent=2))

## 9. Optional: per-trial table for diagnostics

In [ ]:
trials_df = study.trials_dataframe(attrs=('number','value','state','params','duration'))
print(trials_df.head(40).to_string())
trials_df.to_csv(OUTPUT_DIR / 'optuna_trials.csv', index=False)
print(f'\nSaved: {OUTPUT_DIR / "optuna_trials.csv"}')

## After this notebook

1. **Click "Save Version"** in the top-right, choose **"Save & Run All (Commit)"** to run in background. Email arrives at completion (7–12h).
2. When done, the output files are at `/kaggle/working/`:
   - `chemprop_esol_phase5_best_params.json` ← best hyperparameters for Sera 4
   - `optuna_chemprop.db` ← full study, can re-load for resume or for more trials
   - `optuna_trials.csv` ← per-trial table for diagnostics
3. To use these in Sera 4 (5-seed outer evaluation), upload them back to the `qsar-esol-data` dataset as a new version, or create a small companion dataset `qsar-esol-phase5-results`.
4. Sera 4 runs locally (Colab Free can handle 5 single trainings in ~25 min total) — no need to use up Kaggle GPU quota for it.